# Week 12 — Monday: Imbalanced Data

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/12/images/parkes.jpg" width="480" alt="The Parkes radio telescope, a large white dish, against a blue sky.">

In 1967, graduate student Jocelyn Bell's radio telescope printed 96 feet of chart paper a day, and she read it by hand. Weeks into the survey, she noticed

> "a bit of 'scruff' on the records, which did not look exactly like a scintillating source, and yet did not look exactly like man-made interference either."

It was the first **pulsar**: a collapsed star that spins and sweeps a radio beam past Earth like a lighthouse, one pulse every 1⅓ seconds. Today, surveys like the one at the Parkes telescope (above) produce millions of candidate signals. Almost all are interference: phones, radar, satellites.

> 🎯 **Can a model find the rare real pulsars? And what does "accurate" mean when they are rare?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · How rare is a pulsar? (SLO 12A) |
| ~10 min | 2 · 98% accurate, still missing pulsars (SLO 12A) |
| ~8 min | 3 · Moving the threshold (SLO 12A) |
| ~8 min | 4 · Rebalancing the training data with imblearn (SLO 12A) |
| ~5 min | 5 · Rebalance after the split |
| ~4 min | 6 · A toolbox for imbalanced data (SLO 12A) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: HTRU2, R. J. Lyon et al. (2016), *Monthly Notices of the Royal Astronomical Society* 459, 1104–1123; UCI Machine Learning Repository, CC BY 4.0. Quote: S. J. Bell Burnell, "Petit Four," *Annals of the New York Academy of Sciences* 302 (1977). Photo: Amanda Slater, CC BY-SA 2.0, via Wikimedia Commons</sub>

---
## 1 · How Rare Is a Pulsar? (SLO 12A) · ~5 min

One row = one candidate signal from the Parkes survey, checked by a person. Eight numbers describe its shape:

* `profile_...`: the signal stacked up over many turns of the star. A pulsar gives one sharp, narrow spike per turn
* `dmsnr_...`: a real pulsar's signal gets smeared on its long trip through space; a signal made on Earth doesn't
* `pulsar`: **1** = real pulsar, **0** = interference or noise

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_predict, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

candidates = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/pulsars.csv")
X = candidates.drop(columns="pulsar")
y = candidates["pulsar"]
print(y.value_counts())
print(f"share of pulsars: {y.mean():.1%}")
candidates.head()

> 🎯 **Predict First:** A lazy "model" calls every candidate noise. What's its accuracy? How many pulsars does it find?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(f"training: {len(y_train)} candidates, {y_train.sum()} pulsars | test: {len(y_test)} candidates, {y_test.sum()} pulsars")

lazy = np.zeros(len(y_test))                  # 0 = "noise", for every test candidate
print(f"lazy model | accuracy {accuracy_score(y_test, lazy):.3f} | recall {recall_score(y_test, lazy):.3f}")

<details><summary>Answer</summary>

* **90.8% accurate, and it finds 0 of the 410 test pulsars** (Week 6's lazy Chinstrap model, again)
* **Imbalanced data:** one class, the *majority*, far outnumbers the other, the *minority*. Here: 10 noise candidates for every pulsar. Accuracy rewards guessing the majority
* `stratify=y` keeps the same 9.2% of pulsars in the training and the test set. With a small minority, a plain random split can give the test set too many or too few by luck
</details>

---
## 2 · 98% Accurate, Still Missing Pulsars (SLO 12A) · ~10 min

Week 11's logistic regression on all 8 columns, scaled:

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"accuracy {accuracy_score(y_test, y_pred):.3f} | recall {recall_score(y_test, y_pred):.3f}"
      f" | precision {precision_score(y_test, y_pred):.3f}")
pd.DataFrame(confusion_matrix(y_test, y_pred), index=["really noise", "really pulsar"],
             columns=["called noise", "called pulsar"])

<details><summary>Answer</summary>

* **98.0% accurate**, only 7 points above the lazy model
* **Recall 0.827:** it misses **71 of the 410** test pulsars (false negatives)
* **Precision 0.947:** only 19 false alarms. The model is careful about calling "pulsar", and loose about calling "noise"
</details>

Why does it lean that way? An experiment on paper first, with two of the eight columns and a handful of training candidates.

> 📄 **Handout 1 — Same pulsars, more noise.** On each panel, draw the straight line that makes the fewest mistakes. 🗣️ Only noise was added. Why did your line move?

In [ ]:
two = ["profile_mean", "profile_sd"]
colors = {"noise": "#2a78d6", "pulsar": "#eb6834"}
means = np.linspace(0, 200, 50)

pulsars12 = X_train[y_train == 1].sample(12, random_state=8)      # Handout 1's candidates
noise120 = X_train[y_train == 0].sample(120, random_state=40)
for noise in [noise120.head(12), noise120]:
    sample = pd.concat([pulsars12, noise])
    small = LogisticRegression().fit(sample[two], y_train[sample.index])
    b0 = small.intercept_[0]
    b_mean, b_sd = small.coef_[0]
    sample["kind"] = y_train[sample.index].map({0: "noise", 1: "pulsar"})
    small_fig = px.scatter(sample, x="profile_mean", y="profile_sd", color="kind", symbol="kind",
                           color_discrete_map=colors, symbol_map={"noise": "circle", "pulsar": "triangle-up"},
                           range_x=[0, 200], range_y=[25, 85], title=f"12 Pulsars, {len(noise)} Noise: Logistic Regression's 50% Line",
                           labels={"profile_mean": "profile mean", "profile_sd": "profile standard deviation", "kind": ""})
    small_fig.add_scatter(x=means, y=-(b0 + b_mean * means) / b_sd, mode="lines", name="50% line", line_color="black")
    small_fig.show()
    print(f"learned from {len(noise)} noise | calls {(small.predict(pulsars12[two]) == 0).sum()} of the 12 pulsars noise"
          f" | calls {small.predict(noise120[two]).sum()} of all 120 noise candidates pulsars")

<details><summary>Answer</summary>

* **Left, 12 and 12:** a clear gap, and a line through it makes no mistakes
* **Right, 12 and 120:** the noise reaches into the pulsars' corner. A line that catches all 12 pulsars must take at least **7** noise candidates with them: 7 mistakes. Giving up the 3 pulsars inside the noise costs only **3**. Fewest mistakes means giving them up
* Logistic regression agrees. On the right, its line calls 3 of the 12 pulsars noise. The left panel's line, used on all 120 noise candidates, would call 11 of them pulsars
* Nothing about the pulsars changed, only how much noise surrounds them. With 10 noise candidates for every pulsar, a model that learns by "fewest mistakes overall" gives up the pulsars near the noise
</details>

The same with all 13,423 training candidates:

In [ ]:
model2 = LogisticRegression().fit(X_train[two], y_train)
b0 = model2.intercept_[0]
b_mean, b_sd = model2.coef_[0]

fig = px.scatter(X_train, x="profile_mean", y="profile_sd", color=y_train.map({0: "noise", 1: "pulsar"}),
                 color_discrete_map=colors, opacity=0.4, range_x=[0, 200], range_y=[25, 85],
                 title="Training Candidates: Two of the Eight Columns",
                 labels={"profile_mean": "profile mean", "profile_sd": "profile standard deviation", "color": ""})
# The 50% line is where the score b0 + b_mean × mean + b_sd × sd is 0 (Week 11). Solved for sd:
fig.add_scatter(x=means, y=-(b0 + b_mean * means) / b_sd, mode="lines", name="50% line", line_color="black")
fig.show()

> 💬 **Question:** Which class does this line favor?

<details><summary>Answer</summary>

* The noise. It calls **381 of the 1,229** training pulsars noise (31%), and only **62 of the 12,194** noise candidates pulsars (0.5%)
* The handout's right panel, at full size: sliding the line into the noise would catch more pulsars, and misjudge more noise. So the line hugs the minority
</details>

---
## 3 · Moving the Threshold (SLO 12A) · ~8 min

`predict` calls "pulsar" when the probability is **at least 0.5**. That 0.5 is a **threshold**, and we can move it.

In [ ]:
for t, dash in [(0.2, "dash"), (0.1, "dot")]:
    score = np.log(t / (1 - t))      # the score z where the S-curve gives probability t (it was 0 for t = 0.5)
    fig.add_scatter(x=means, y=(score - b0 - b_mean * means) / b_sd, mode="lines", name=f"threshold {t}",
                    line=dict(color="black", dash=dash))
fig.show()

<details><summary>Answer</summary>

* A lower threshold slides the line **into the noise**: it catches more pulsars, and more noise with them
* The line keeps its tilt. Only where it sits changes
</details>

Back to all 8 columns. Choosing a threshold is choosing a knob, so we choose it with **cross-validation on the training set**, not by peeking at the test set (Week 7). `cross_val_predict` gives each training candidate a probability from a model that didn't learn from it:

In [ ]:
p_train = cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in np.arange(0.02, 0.99, 0.02):
    called = p_train >= t
    rows.append({"threshold": t, "recall": recall_score(y_train, called), "precision": precision_score(y_train, called)})
curve = pd.DataFrame(rows)
fig = px.line(curve, x="threshold", y=["recall", "precision"], labels={"value": "", "variable": ""},
              title="Logistic Regression, 8 Columns: Recall and Precision by Threshold (5-Fold CV)")
fig.show()

> 📄 **Handout 2 — Choose a threshold.** Mark one threshold for each team. 🗣️ Who should choose it: the data scientist or the astronomers?

<details><summary>Answer</summary>

* Moving the threshold trades one mistake for the other: **lower = higher recall, lower precision**
* **Team A** (two people checking candidates by eye) can't afford many false alarms: a middle threshold, around 0.3–0.5
* **Team B** (a complete census) wants recall: around 0.05–0.1, and lots of checking
* There is no correct threshold in the data. It comes from the **costs** of each mistake, which the astronomers know. The data scientist shows the trade-off
</details>

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
for t in [0.5, 0.1]:
    called = p_test >= t
    print(f"threshold {t} | recall {recall_score(y_test, called):.3f} | precision {precision_score(y_test, called):.3f}"
          f" | flagged {called.sum()} of {len(y_test)}")

<details><summary>Answer</summary>

* At 0.1: recall **0.827 → 0.902** (31 more pulsars found); precision **0.947 → 0.797**; 464 candidates to check instead of 358
</details>

> 📌 **One split:** 410 test pulsars. One pulsar is a quarter of a point of recall.

---
## 4 · Rebalancing the Training Data with imblearn (SLO 12A) · ~8 min

Instead of moving the threshold after training, change what the model **learns from**:

| method | what it does |
|---|---|
| **class weights** | each pulsar counts about 10× in the model's mistakes: `class_weight="balanced"` |
| **random oversampling** | copy random pulsars until there are as many pulsars as noise |
| **random undersampling** | throw away random noise candidates until there are as few noise as pulsars |
| **SMOTE** | make **new**, synthetic pulsars on the line between a pulsar and a nearby pulsar |

Class weights are an option of the model itself. The other three come from **imbalanced-learn** (`import imblearn`), a library built on scikit-learn just for imbalanced data. Colab has it; elsewhere, `pip install imbalanced-learn`. Each method is a **sampler** with one job, `fit_resample(X, y)`: it returns a new `X` and `y`, rebalanced.

> 📄 **Handout 3 — Make synthetic pulsars.** Do SMOTE by hand.

In [ ]:
# imbalanced-learn (imblearn) comes with Colab; elsewhere: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline

X_two_smote, y_two_smote = SMOTE(random_state=1).fit_resample(X_train[two], y_train)
print("before:", y_train.value_counts().to_dict(), "| after SMOTE:", y_two_smote.value_counts().to_dict())

synthetic = X_two_smote.iloc[len(X_train):]       # SMOTE adds its new pulsars after the real candidates
fig = px.scatter(X_train, x="profile_mean", y="profile_sd", color=y_train.map({0: "noise", 1: "pulsar"}),
                 color_discrete_map=colors, opacity=0.3, range_x=[0, 200], range_y=[25, 85], title="SMOTE's Synthetic Pulsars (500 of Them)",
                 labels={"profile_mean": "profile mean", "profile_sd": "profile standard deviation", "color": ""})
sample = synthetic.sample(500, random_state=1)
fig.add_scatter(x=sample["profile_mean"], y=sample["profile_sd"], mode="markers", name="synthetic pulsar",
                marker=dict(color="black", size=4))
fig.show()

<details><summary>Answer</summary>

* SMOTE made 10,965 new pulsars, so the training set is now 50/50
* Most land among real pulsars. Some land in the **noise**: SMOTE assumes the space between two pulsars is pulsar territory, and it isn't always
</details>

Which rebalancing works best? Cross-validation on the training set. `imblearn` has its own `make_pipeline`, which works like scikit-learn's with one difference: a sampler runs only when the pipeline **learns** (`fit`), never when it predicts. So in cross-validation, only the folds it learns from get rebalanced, and each held-out fold keeps the real mix.

In [ ]:
options = {
    "nothing": make_pipeline(StandardScaler(), LogisticRegression()),
    "class weights": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
    "oversample (copy pulsars)": make_imb_pipeline(StandardScaler(), RandomOverSampler(random_state=1), LogisticRegression()),
    "undersample (drop noise)": make_imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=1), LogisticRegression()),
    "SMOTE": make_imb_pipeline(StandardScaler(), SMOTE(random_state=1), LogisticRegression()),
}
rows = []
for name, option in options.items():
    scores = cross_validate(option, X_train, y_train, cv=5, scoring=["recall", "precision"])
    rows.append({"rebalancing": name, "recall": scores["test_recall"].mean(), "precision": scores["test_precision"].mean()})
pd.DataFrame(rows).round(3)

> 💬 **Question:** All four land in the same spot. Where have we seen recall ≈ 0.91 with precision ≈ 0.79 before?

<details><summary>Answer</summary>

* On Part 3's curve, near **threshold 0.1**. For logistic regression, rebalancing mostly slides the line, like moving the threshold
* Undersampling did just as well with a tenth of the noise, but it threw information away; with fewer rows, that would hurt
* A rebalanced model thinks pulsars are half the sky, so its probabilities are too high. Read them as scores, not real chances
* Rebalancing earns its keep with models that don't give probabilities, with very rare classes, and with GridSearchCV: tune with `scoring="recall"` (or `"f1"`, which is high only when precision and recall both are), never the default accuracy
</details>

---
## 5 · Rebalance After the Split · ~5 min

A common mistake: SMOTE the **whole** dataset, then split it. Here with Week 11's random forest:

> 🎯 **Predict First:** Which order gives the higher test score? Which score would you believe?

In [ ]:
# Wrong order: synthetic pulsars from ALL candidates, then split
X_all, y_all = SMOTE(random_state=1).fit_resample(X, y)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.25, random_state=1, stratify=y_all)
forest = RandomForestClassifier(random_state=1).fit(X_tr, y_tr)
y_hat = forest.predict(X_te)
print(f"SMOTE, then split | recall {recall_score(y_te, y_hat):.3f} | precision {precision_score(y_te, y_hat):.3f}"
      f" | test set {y_te.mean():.0%} pulsars")

# Right order: split first; the imblearn pipeline makes synthetic pulsars from the training set only
smote_forest = make_imb_pipeline(SMOTE(random_state=1), RandomForestClassifier(random_state=1))
smote_forest.fit(X_train, y_train)
y_hat = smote_forest.predict(X_test)
print(f"split, then SMOTE | recall {recall_score(y_test, y_hat):.3f} | precision {precision_score(y_test, y_hat):.3f}"
      f" | test set {y_test.mean():.0%} pulsars")

<details><summary>Answer</summary>

* **Wrong order:** recall 0.971, precision 0.984. Most test pulsars are synthetic, made from training pulsars: the forest has nearly seen them. The score measures **memory**, like scoring a model on the rows it learned from (Week 6). Overfitting, hidden by a leak
* And that test set is **50% pulsars**. The sky is 9%: its precision says nothing about real use
* **Right order:** recall 0.890, precision 0.863. That's the one to believe
* **Rule:** rebalance **only the training data**, after the split. The `imblearn` pipeline does it for you, in cross-validation and in the final fit
</details>

Rebalancing keeps the label and changes how much each example counts. What if the label you care about is too rare to learn at all?

> "When Virginia Eubanks studied a method for assessing a child's risk of being abused in Pennsylvania's Allegheny County, she found that cases of child maltreatment-related fatalities and near fatalities, which are of greatest concern to the state, are extremely rare. To produce useful results, the county's predictive model had to optimize on more common outcomes. In the end, two proxies for child harm were used: the likelihood that another call will be made about the child to the abuse and neglect hotline, and the likelihood that the child will be placed in foster care."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** The county swapped a rare label for a common one. Besides harm, what does "another call to the hotline" measure? Whose families get flagged, and whose never show up in the data?

<details><summary>Answer</summary>

* A call measures who gets **reported**. Calls and foster placements follow contact with public services, so they track **poverty** as much as harm
* Poor families were over-represented; suburban families, who handle problems through private doctors and insurance, were "nowhere to be seen in the data" (ch. 3)
* A pulsar label is checked by an astronomer. A proxy label quietly changes the question, and the model answers the new question very well
</details>

---
## 6 · A Toolbox for Imbalanced Data (SLO 12A) · ~4 min

Does rebalancing work the same way for every model? Week 11's random forest, four ways. The last one is from `imblearn` too: `BalancedRandomForestClassifier` is Week 11's bagging, where each tree learns from its own **balanced** sample: pulsars and an equal number of noise candidates, both drawn with replacement. (About 15 seconds to run.)

> 🎯 **Predict First:** which of the four will change the forest the least?

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

forests = {
    "nothing": RandomForestClassifier(random_state=1),
    "class weights": RandomForestClassifier(class_weight="balanced", random_state=1),
    "SMOTE": make_imb_pipeline(SMOTE(random_state=1), RandomForestClassifier(random_state=1)),
    # imblearn's newer defaults, written out so every version builds the same forest
    "balanced forest": BalancedRandomForestClassifier(sampling_strategy="all", replacement=True, bootstrap=False, random_state=1),
}
for name, forest in forests.items():
    y_hat = forest.fit(X_train, y_train).predict(X_test)
    print(f"{name:15} | recall {recall_score(y_test, y_hat):.3f} | precision {precision_score(y_test, y_hat):.3f} | flagged {y_hat.sum()}")

<details><summary>Answer</summary>

* **Class weights barely move the forest.** Its deep trees end in pure leaves, so the weights hardly change their votes. For logistic regression (Part 4) they worked as well as SMOTE. A tool that helps one model can do nothing for another: compare them with CV
* **SMOTE and the balanced forest** trade precision for recall, like a lower threshold: recall 0.851 → 0.890 and 0.900, precision 0.948 → 0.863 and 0.813
* The balanced forest is also the fastest: each tree learns from about 2,500 candidates, not 13,423. Each tree throws most of the noise away, but every tree throws away a different part
</details>

> 📌 **One split:** one test pulsar is a quarter of a point of recall.

**The toolbox.** The first three cost nothing: always start there. The rest are knobs: compare them with cross-validation, scored with `"recall"`, `"f1"` or the real costs of each mistake.

| | strategy | in code | keep in mind |
|---|---|---|---|
| 1 | **Look** at the class shares, and split with `stratify` | `y.value_counts(normalize=True)`, `train_test_split(..., stratify=y)` | a plain random split can give the test set too many or too few of the minority |
| 2 | **Measure** recall, precision and F1, next to the lazy model | `recall_score`, `precision_score`, `f1_score`, `confusion_matrix`; `GridSearchCV(..., scoring="f1")` | accuracy rewards the majority |
| 3 | **Move the threshold** | `predict_proba(X)[:, 1] >= t`, with `t` from `cross_val_predict` | choose it with CV and the costs of each mistake, never on the test set |
| 4 | **Class weights** | `class_weight="balanced"` in `LogisticRegression`, trees, forests | not every model has it (kNN doesn't); barely moves deep trees |
| 5 | **Random oversampling** | `imblearn.over_sampling.RandomOverSampler` | exact copies: a deep tree memorizes them |
| 6 | **Random undersampling** | `imblearn.under_sampling.RandomUnderSampler` | throws data away; fine when the majority is huge |
| 7 | **SMOTE** | `imblearn.over_sampling.SMOTE` (`SMOTENC` for word columns) | synthetic rows can land in the majority's territory |
| 8 | **Balanced ensembles** | `imblearn.ensemble.BalancedRandomForestClassifier` | bagging with undersampling: every tree sees a different part of the majority |
| 9 | **Rebalance after the split** | `imblearn.pipeline.make_pipeline(scaler, sampler, model)` | rebalancing before the split leaks |
| 10 | **Question the label** | — | a common stand-in for a rare outcome changes the question |

After any rebalancing (4–8), the model thinks the classes are equally common, so its probabilities run too high: read them as scores, not real chances.

---
### Wrap-up

1. **Imbalanced data:** check the class shares first. Accuracy rewards the majority, so report **recall and precision**, and split with `stratify=y`
2. A model that minimizes mistakes **leans toward the majority**: near the noise, each pulsar it catches costs several false alarms
3. **The threshold is a knob:** lower → more found, more false alarms. Choose it with CV and the real costs of each mistake, never on the test set 📌
4. **Rebalancing** (class weights; `imblearn`'s oversampling, undersampling, SMOTE and balanced forests) shifts the balance during training. For logistic regression, much like moving the threshold; for a forest, class weights do almost nothing
5. Rebalance **only the training data**, after the split: the `imblearn` pipeline does it for you
6. Part 6's table is the checklist for any problem where one class is rare

**Wednesday:** time-series data, where a random split lets the model peek at the future.